## Cell 1 — Import library

In [1]:
import pandas as pd
from pathlib import Path

## Cell 2 — Tentukan folder project

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw dir:", RAW_DIR)
print("Processed dir:", PROCESSED_DIR)
print("Output tables dir:", OUTPUT_TABLES_DIR)

Project root: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL
Raw dir: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\raw
Processed dir: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\processed
Output tables dir: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables


## Cell 3 — Cari file Excel log mentah

In [3]:
excel_files = sorted(RAW_DIR.glob("*.xlsx"))

print("Jumlah file Excel ditemukan:", len(excel_files))

for file in excel_files:
    print("-", file.name)

Jumlah file Excel ditemukan: 2
- logs_CAK1CAB3-IF-49-02_20260122-1521.xlsx
- logs_CAK1CAB3-IF-49-03_20260122-1521.xlsx


## Cell 4 — Baca dan gabungkan raw log

In [4]:
log_02 = pd.read_excel(excel_files[0])
log_03 = pd.read_excel(excel_files[1])

log_02["kelas"] = "IF-49-02"
log_03["kelas"] = "IF-49-03"

log_raw = pd.concat([log_02, log_03], ignore_index=True)

print("Ukuran log_02:", log_02.shape)
print("Ukuran log_03:", log_03.shape)
print("Ukuran log_raw gabungan:", log_raw.shape)

display(log_raw.head())

Ukuran log_02: (89454, 10)
Ukuran log_03: (76192, 10)
Ukuran log_raw gabungan: (165646, 10)


,Time,User full name,Affected user,Event context,Component,Event name,Description,Origin,IP address,kelas
0,"22/01/26, 15:21:30",ADE KURNIAWAN,-,Course: KALKULUS IF-49-02 [DLW],Logs,Log report viewed,The user with id &#039;54114&#039; viewed the ...,web,103.233.100.203,IF-49-02
1,"22/01/26, 15:20:55",ADE KURNIAWAN,-,Course: KALKULUS IF-49-02 [DLW],System,Course viewed,The user with id &#039;54114&#039; viewed the ...,web,103.233.100.203,IF-49-02
2,"20/01/26, 15:53:58",HASNAH MUSHAIDAH SOFIA,-,Course: KALKULUS IF-49-02 [DLW],System,Course viewed,The user with id &#039;41146&#039; viewed the ...,web,103.233.100.204,IF-49-02
3,"20/01/26, 15:14:42",HASNAH MUSHAIDAH SOFIA,-,Course: KALKULUS IF-49-02 [DLW],Logs,Log report viewed,The user with id &#039;41146&#039; viewed the ...,web,103.233.100.204,IF-49-02
4,"20/01/26, 15:09:21",HASNAH MUSHAIDAH SOFIA,-,Course: KALKULUS IF-49-02 [DLW],Logs,Log report viewed,The user with id &#039;41146&#039; viewed the ...,web,103.233.100.204,IF-49-02


## Cell 5 — Baca label performa mahasiswa

In [5]:
LABEL_PATH = PROCESSED_DIR / "02_label_performance.csv"

label_df = pd.read_csv(LABEL_PATH, dtype={"nim": str})

print("Ukuran label_df:", label_df.shape)
display(label_df.head())

print("\nDistribusi kelas:")
display(label_df["kelas"].value_counts())

print("\nDistribusi label performance:")
display(label_df["label_performance"].value_counts())

Ukuran label_df: (89, 7)


,no,nim,nama,kelas,nilai_total,nilai_indeks,label_performance
0,1,103012400054,MUHAMMAD FAZLI AULIA,IF-49-02,11.08,E,Rendah
1,2,103012400409,FACHRI RAZAQ KHALISH,IF-49-02,6.38,E,Rendah
2,3,103012500035,RIZKY ALVINO PUTRA,IF-49-02,62.96,BC,Sedang
3,4,103012500050,ALIF ZAIDAN ZIDNA FANN,IF-49-02,67.76,B,Sedang
4,5,103012500054,RAFI AJRUR MUZAKKI,IF-49-02,62.60,BC,Sedang



Distribusi kelas:


kelas
IF-49-02    45
IF-49-03    44
Name: count, dtype: int64


Distribusi label performance:


label_performance
Sedang    54
Tinggi    20
Rendah    15
Name: count, dtype: int64

## Cell 6 — Konversi Time menjadi timestamp

In [6]:
log_raw["timestamp"] = pd.to_datetime(
    log_raw["Time"],
    format="%d/%m/%y, %H:%M:%S",
    errors="coerce"
)

print("Jumlah timestamp gagal dibaca:", log_raw["timestamp"].isna().sum())
print("Timestamp awal:", log_raw["timestamp"].min())
print("Timestamp akhir:", log_raw["timestamp"].max())

Jumlah timestamp gagal dibaca: 0
Timestamp awal: 2025-09-16 09:49:33
Timestamp akhir: 2026-01-22 15:21:32


## Cell 7 — Buat fungsi normalisasi nama

In [7]:
def normalize_name(name):
    if pd.isna(name):
        return ""
    
    name = str(name)
    name = name.upper()
    name = " ".join(name.split())
    
    return name

## Cell 8 — Buat kolom nama normalisasi

In [8]:
log_raw["user_key"] = log_raw["User full name"].apply(normalize_name)
label_df["nama_key"] = label_df["nama"].apply(normalize_name)

print("Contoh user_key dari log:")
display(log_raw[["User full name", "user_key"]].head())

print("\nContoh nama_key dari label:")
display(label_df[["nama", "nama_key"]].head())

Contoh user_key dari log:


,User full name,user_key
0,ADE KURNIAWAN,ADE KURNIAWAN
1,ADE KURNIAWAN,ADE KURNIAWAN
2,HASNAH MUSHAIDAH SOFIA,HASNAH MUSHAIDAH SOFIA
3,HASNAH MUSHAIDAH SOFIA,HASNAH MUSHAIDAH SOFIA
4,HASNAH MUSHAIDAH SOFIA,HASNAH MUSHAIDAH SOFIA



Contoh nama_key dari label:


,nama,nama_key
0,MUHAMMAD FAZLI AULIA,MUHAMMAD FAZLI AULIA
1,FACHRI RAZAQ KHALISH,FACHRI RAZAQ KHALISH
2,RIZKY ALVINO PUTRA,RIZKY ALVINO PUTRA
3,ALIF ZAIDAN ZIDNA FANN,ALIF ZAIDAN ZIDNA FANN
4,RAFI AJRUR MUZAKKI,RAFI AJRUR MUZAKKI


## Cell 9 — Cek nama mahasiswa label yang muncul di log

In [9]:
label_names = set(label_df["nama_key"])
log_names = set(log_raw["user_key"])

matched_names = label_names.intersection(log_names)
label_not_in_log = label_names - log_names
log_not_in_label = log_names - label_names

print("Jumlah nama mahasiswa di label:", len(label_names))
print("Jumlah user unik di log:", len(log_names))
print("Jumlah nama label yang ditemukan di log:", len(matched_names))
print("Jumlah nama label yang TIDAK ditemukan di log:", len(label_not_in_log))
print("Jumlah user log yang tidak ada di label:", len(log_not_in_label))

print("\nNama label yang tidak ditemukan di log:")
for name in sorted(label_not_in_log):
    print("-", name)

Jumlah nama mahasiswa di label: 89
Jumlah user unik di log: 96
Jumlah nama label yang ditemukan di log: 88
Jumlah nama label yang TIDAK ditemukan di log: 1
Jumlah user log yang tidak ada di label: 8

Nama label yang tidak ditemukan di log:
- VANIA


## Cell 9A — Cari nama log yang mengandung “VAN”

In [10]:
candidate_vania = (
    log_raw[
        log_raw["user_key"].str.contains("VAN", na=False)
    ][["User full name", "user_key", "kelas"]]
    .drop_duplicates()
    .sort_values(["kelas", "user_key"])
)

print("Jumlah kandidat nama mengandung 'VAN':", len(candidate_vania))
display(candidate_vania)

Jumlah kandidat nama mengandung 'VAN': 1


,User full name,user_key,kelas
1167,VANIA VANIA,VANIA VANIA,IF-49-02


## Cell 9B — Cari nama yang mirip dengan VANIA

In [11]:
from difflib import get_close_matches

similar_to_vania = get_close_matches(
    "VANIA",
    sorted(log_names),
    n=20,
    cutoff=0.3
)

print("Nama log yang mirip dengan VANIA:")
for name in similar_to_vania:
    print("-", name)

Nama log yang mirip dengan VANIA:
- VANIA VANIA
- RAYNISA ZAHRA
- ADE KURNIAWAN
- ZAKY NAUFAL
- OLIVIA CAHYA DANISWARA
- SYIFA KHAIRUNNISA
- RIESPATI KHAIRUNNISA
- EZZAR KAYSAN GUMILAR
- ADY RAHMAN WICAKSONO
- ALFITO MAULANA


## Cell 9C — Cek data label VANIA

In [12]:
label_df[label_df["nama_key"] == "VANIA"]

,no,nim,nama,kelas,nilai_total,nilai_indeks,label_performance,nama_key
37,38,103012500407,VANIA,IF-49-02,63.65,BC,Sedang,VANIA


## Cell 9D — Manual mapping nama

In [13]:
manual_name_mapping = {
    "VANIA": "VANIA VANIA"
}

label_df["match_key"] = label_df["nama_key"].replace(manual_name_mapping)
log_raw["match_key"] = log_raw["user_key"]

display(label_df[label_df["nama_key"] == "VANIA"][[
    "nim", "nama", "kelas", "nilai_total", "nilai_indeks", "label_performance", "nama_key", "match_key"
]])

,nim,nama,kelas,nilai_total,nilai_indeks,label_performance,nama_key,match_key
37,103012500407,VANIA,IF-49-02,63.65,BC,Sedang,VANIA,VANIA VANIA


## Cell 9E — Cek ulang matching setelah mapping

In [14]:
label_names = set(label_df["match_key"])
log_names = set(log_raw["match_key"])

matched_names = label_names.intersection(log_names)
label_not_in_log = label_names - log_names
log_not_in_label = log_names - label_names

print("Jumlah nama mahasiswa di label:", len(label_names))
print("Jumlah user unik di log:", len(log_names))
print("Jumlah nama label yang ditemukan di log:", len(matched_names))
print("Jumlah nama label yang TIDAK ditemukan di log:", len(label_not_in_log))
print("Jumlah user log yang tidak ada di label:", len(log_not_in_label))

print("\nNama label yang tidak ditemukan di log:")
for name in sorted(label_not_in_log):
    print("-", name)

Jumlah nama mahasiswa di label: 89
Jumlah user unik di log: 96
Jumlah nama label yang ditemukan di log: 89
Jumlah nama label yang TIDAK ditemukan di log: 0
Jumlah user log yang tidak ada di label: 7

Nama label yang tidak ditemukan di log:


## Cell 10 — Filter log hanya mahasiswa resmi

In [15]:
log_students = log_raw[log_raw["match_key"].isin(label_names)].copy()

print("Ukuran raw log sebelum filter:", log_raw.shape)
print("Ukuran log setelah filter mahasiswa:", log_students.shape)

print("\nJumlah user unik setelah filter:", log_students["match_key"].nunique())

print("\nDistribusi event setelah filter berdasarkan kelas:")
display(log_students["kelas"].value_counts())

Ukuran raw log sebelum filter: (165646, 13)
Ukuran log setelah filter mahasiswa: (156145, 13)

Jumlah user unik setelah filter: 89

Distribusi event setelah filter berdasarkan kelas:


kelas
IF-49-02    84418
IF-49-03    71727
Name: count, dtype: int64

## Cell 11 — Lihat user yang dibuang

In [16]:
removed_users = sorted(log_not_in_label)

print("Jumlah user raw log yang tidak ada di label:", len(removed_users))

for user in removed_users:
    print("-", user)

Jumlah user raw log yang tidak ada di label: 7
- -
- ADE KURNIAWAN
- ADMIN CELOE LMS
- DEDE TARWIDI
- GUEST USER
- HASNAH MUSHAIDAH SOFIA
- WS USER


## Cell 12 — Bentuk event log dasar sesuai proposal

In [17]:
event_log = log_students.copy()

event_log["case_id"] = event_log["match_key"]
event_log["activity"] = event_log["Event name"]

event_log = event_log[[
    "case_id",
    "activity",
    "timestamp",
    "kelas",
    "Component",
    "Event context",
    "Description",
    "Origin",
    "IP address"
]].copy()

event_log = event_log.sort_values(
    by=["case_id", "timestamp"]
).reset_index(drop=True)

display(event_log.head(10))

print("Ukuran event_log:", event_log.shape)
print("Jumlah case_id unik:", event_log["case_id"].nunique())
print("Jumlah activity unik:", event_log["activity"].nunique())

,case_id,activity,timestamp,kelas,Component,Event context,Description,Origin,IP address
0,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 13:30:09,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,103.233.100.204
1,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 13:31:17,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,103.233.100.203
2,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 13:39:55,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,103.233.100.202
3,ADITYA PUTRA PERMANA,Course activity completion updated,2025-09-18 14:24:42,IF-49-02,System,File: Materi 1 Sistem Bilangan Riil dan Pertak...,The user with id &#039;124462&#039; updated th...,web,103.233.100.202
4,ADITYA PUTRA PERMANA,Course activity completion updated,2025-09-18 14:24:42,IF-49-02,System,File: Materi 1 Sistem Bilangan Riil dan Pertak...,The user with id &#039;124462&#039; updated th...,web,103.233.100.202
5,ADITYA PUTRA PERMANA,Course module viewed,2025-09-18 14:24:42,IF-49-02,File,File: Materi 1 Sistem Bilangan Riil dan Pertak...,The user with id &#039;124462&#039; viewed the...,web,103.233.100.202
6,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 15:41:24,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,103.233.100.202
7,ADITYA PUTRA PERMANA,Course module viewed,2025-09-18 15:41:57,IF-49-02,File,File: Materi 1 Sistem Bilangan Riil dan Pertak...,The user with id &#039;124462&#039; viewed the...,web,103.233.100.202
8,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 16:40:55,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,36.72.198.4
9,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 16:49:33,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,36.72.198.4


Ukuran event_log: (156145, 9)
Jumlah case_id unik: 89
Jumlah activity unik: 29


## Cell 13 — Cek missing value event log

In [18]:
missing_event_log = event_log.isna().sum().sort_values(ascending=False)

display(missing_event_log)

case_id          0
activity         0
timestamp        0
kelas            0
Component        0
Event context    0
Description      0
Origin           0
IP address       0
dtype: int64

## Cell 14 — Ringkasan preprocessing awal

In [19]:
summary_preprocessing = pd.DataFrame({
    "tahap": [
        "Raw log gabungan",
        "Setelah filter mahasiswa resmi"
    ],
    "jumlah_event": [
        len(log_raw),
        len(event_log)
    ],
    "jumlah_user_unik": [
        log_raw["match_key"].nunique(),
        event_log["case_id"].nunique()
    ],
    "jumlah_activity_unik": [
        log_raw["Event name"].nunique(),
        event_log["activity"].nunique()
    ]
})

display(summary_preprocessing)

,tahap,jumlah_event,jumlah_user_unik,jumlah_activity_unik
0,Raw log gabungan,165646,96,63
1,Setelah filter mahasiswa resmi,156145,89,29


## Cell 15 — Tambahkan label ke event log

In [20]:
label_for_join = label_df[[
    "match_key",
    "nim",
    "nama",
    "nilai_total",
    "nilai_indeks",
    "label_performance"
]].copy()

event_log_labeled = event_log.merge(
    label_for_join,
    left_on="case_id",
    right_on="match_key",
    how="left"
)

event_log_labeled = event_log_labeled.drop(columns=["match_key"])

display(event_log_labeled.head(10))

print("Ukuran event_log_labeled:", event_log_labeled.shape)
print("Jumlah case_id unik:", event_log_labeled["case_id"].nunique())

print("\nMissing label setelah join:")
display(event_log_labeled[["nim", "nilai_total", "nilai_indeks", "label_performance"]].isna().sum())

print("\nDistribusi event berdasarkan label performance:")
display(event_log_labeled["label_performance"].value_counts())

,case_id,activity,timestamp,kelas,Component,Event context,Description,Origin,IP address,nim,nama,nilai_total,nilai_indeks,label_performance
0,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 13:30:09,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,103.233.100.204,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang
1,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 13:31:17,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,103.233.100.203,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang
2,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 13:39:55,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,103.233.100.202,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang
3,ADITYA PUTRA PERMANA,Course activity completion updated,2025-09-18 14:24:42,IF-49-02,System,File: Materi 1 Sistem Bilangan Riil dan Pertak...,The user with id &#039;124462&#039; updated th...,web,103.233.100.202,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang
4,ADITYA PUTRA PERMANA,Course activity completion updated,2025-09-18 14:24:42,IF-49-02,System,File: Materi 1 Sistem Bilangan Riil dan Pertak...,The user with id &#039;124462&#039; updated th...,web,103.233.100.202,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang
5,ADITYA PUTRA PERMANA,Course module viewed,2025-09-18 14:24:42,IF-49-02,File,File: Materi 1 Sistem Bilangan Riil dan Pertak...,The user with id &#039;124462&#039; viewed the...,web,103.233.100.202,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang
6,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 15:41:24,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,103.233.100.202,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang
7,ADITYA PUTRA PERMANA,Course module viewed,2025-09-18 15:41:57,IF-49-02,File,File: Materi 1 Sistem Bilangan Riil dan Pertak...,The user with id &#039;124462&#039; viewed the...,web,103.233.100.202,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang
8,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 16:40:55,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,36.72.198.4,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang
9,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 16:49:33,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,36.72.198.4,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang


Ukuran event_log_labeled: (156145, 14)
Jumlah case_id unik: 89

Missing label setelah join:


nim                  0
nilai_total          0
nilai_indeks         0
label_performance    0
dtype: int64


Distribusi event berdasarkan label performance:


label_performance
Sedang    99497
Tinggi    40739
Rendah    15909
Name: count, dtype: int64

## Cell 16 — Simpan event log awal

In [21]:
EVENT_LOG_OUTPUT_PATH = PROCESSED_DIR / "03_event_log_students_initial.csv"
EVENT_LOG_LABELED_OUTPUT_PATH = PROCESSED_DIR / "03_event_log_students_labeled_initial.csv"

event_log.to_csv(EVENT_LOG_OUTPUT_PATH, index=False)
event_log_labeled.to_csv(EVENT_LOG_LABELED_OUTPUT_PATH, index=False)

summary_preprocessing.to_csv(
    OUTPUT_TABLES_DIR / "03_summary_preprocessing_initial.csv",
    index=False
)

print("Event log awal berhasil disimpan ke:")
print(EVENT_LOG_OUTPUT_PATH)

print("\nEvent log berlabel berhasil disimpan ke:")
print(EVENT_LOG_LABELED_OUTPUT_PATH)

print("\nRingkasan preprocessing berhasil disimpan ke:")
print(OUTPUT_TABLES_DIR / "03_summary_preprocessing_initial.csv")

Event log awal berhasil disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\processed\03_event_log_students_initial.csv

Event log berlabel berhasil disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\processed\03_event_log_students_labeled_initial.csv

Ringkasan preprocessing berhasil disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables\03_summary_preprocessing_initial.csv


## Cell 17 — Lihat aktivitas unik setelah filter mahasiswa

In [22]:
activity_counts_initial = (
    event_log_labeled["activity"]
    .value_counts()
    .reset_index()
)

activity_counts_initial.columns = ["activity", "jumlah_event"]
activity_counts_initial["persentase"] = (
    activity_counts_initial["jumlah_event"] / len(event_log_labeled) * 100
).round(2)

display(activity_counts_initial)

,activity,jumlah_event,persentase
0,Quiz attempt viewed,50802,32.54
1,Quiz attempt updated,49343,31.60
2,Course module viewed,19438,12.45
3,Course viewed,10456,6.70
4,Course activity completion updated,6286,4.03
5,Quiz attempt auto-saved,5365,3.44
6,Quiz access was prevented,3521,2.25
7,Quiz attempt summary viewed,2392,1.53
8,Quiz attempt started,2259,1.45
9,Quiz attempt submitted,2210,1.42


## Cell 18 — Lihat kombinasi Component dan Activity

In [23]:
component_activity_counts = (
    event_log_labeled
    .groupby(["Component", "activity"])
    .size()
    .reset_index(name="jumlah_event")
    .sort_values("jumlah_event", ascending=False)
)

display(component_activity_counts.head(50))

,Component,activity,jumlah_event
22,Quiz,Quiz attempt viewed,50802
21,Quiz,Quiz attempt updated,49343
16,Quiz,Course module viewed,10536
28,System,Course viewed,10456
26,System,Course activity completion updated,6286
17,Quiz,Quiz attempt auto-saved,5365
7,File,Course module viewed,3623
23,Safe Exam Browser access rules,Quiz access was prevented,3521
14,H5P,Course module viewed,2614
20,Quiz,Quiz attempt summary viewed,2392


## Cell 19 — Cek activity berdasarkan label performa

In [24]:
activity_by_label = pd.crosstab(
    event_log_labeled["activity"],
    event_log_labeled["label_performance"]
)

activity_by_label["total"] = activity_by_label.sum(axis=1)
activity_by_label = activity_by_label.sort_values("total", ascending=False)

display(activity_by_label)

label_performance,Rendah,Sedang,Tinggi,total
activity,,,,
Quiz attempt viewed,5048,33093,12661,50802
Quiz attempt updated,4917,32070,12356,49343
Course module viewed,2105,11953,5380,19438
Course viewed,1048,6064,3344,10456
Course activity completion updated,618,3957,1711,6286
Quiz attempt auto-saved,377,3112,1876,5365
Quiz access was prevented,500,2268,753,3521
Quiz attempt summary viewed,287,1542,563,2392
Quiz attempt started,275,1451,533,2259


## Cell 20 — Hitung jumlah event per mahasiswa

In [25]:
events_per_student = (
    event_log_labeled
    .groupby(["case_id", "label_performance"])
    .size()
    .reset_index(name="jumlah_event")
)

display(events_per_student.head())

print("Ringkasan jumlah event per mahasiswa berdasarkan label:")
display(
    events_per_student
    .groupby("label_performance")["jumlah_event"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

,case_id,label_performance,jumlah_event
0,ADITYA PUTRA PERMANA,Sedang,2617
1,ADY RAHMAN WICAKSONO,Sedang,1549
2,AGUNG KALEB SASAUW,Tinggi,1809
3,ALFITO MAULANA,Sedang,1277
4,ALFONSUS LIGUORI DANGO,Rendah,1050


Ringkasan jumlah event per mahasiswa berdasarkan label:


,count,mean,median,min,max
label_performance,,,,,
Rendah,15,1060.60,1177.0,68,1890
Sedang,54,1842.54,1724.0,894,3318
Tinggi,20,2036.95,1965.5,1137,3477


## Cell 21 — Simpan tabel inspeksi awal

In [26]:
activity_counts_initial.to_csv(
    OUTPUT_TABLES_DIR / "03_activity_counts_initial.csv",
    index=False
)

component_activity_counts.to_csv(
    OUTPUT_TABLES_DIR / "03_component_activity_counts_initial.csv",
    index=False
)

events_per_student.to_csv(
    OUTPUT_TABLES_DIR / "03_events_per_student_initial.csv",
    index=False
)

print("Tabel inspeksi awal berhasil disimpan ke:")
print(OUTPUT_TABLES_DIR)

Tabel inspeksi awal berhasil disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables


## Cell 22 — Buat fungsi activity mapping

In [36]:
def map_activity(row):
    component = str(row["Component"]).strip()
    activity = str(row["activity"]).strip()

    # Course
    if activity == "Course viewed":
        return "Course Viewed"

    # Course module viewed perlu dibaca bersama Component
    if activity == "Course module viewed":
        if component == "Quiz":
            return "Quiz Module Viewed"
        elif component in ["File", "H5P", "URL", "Page"]:
            return "Material Viewed"
        elif component == "Assignment":
            return "Assignment Viewed"
        elif component == "Forum":
            return "Forum Viewed"
        else:
            return "Module Viewed"

    # Quiz
    if activity == "Quiz attempt viewed":
        return "Quiz Viewed"
    elif activity == "Quiz attempt updated":
        return "Quiz Updated"
    elif activity == "Quiz attempt auto-saved":
        return "Quiz Auto-saved"
    elif activity == "Quiz attempt summary viewed":
        return "Quiz Summary Viewed"
    elif activity == "Quiz attempt started":
        return "Quiz Started"
    elif activity == "Quiz attempt submitted":
        return "Quiz Submitted"
    elif activity == "Quiz access was prevented":
        return "Quiz Access Prevented"

    # Assignment / submission
    elif activity == "Submission form viewed.":
        return "Assignment Form Viewed"
    elif activity == "The status of the submission has been viewed.":
        return "Assignment Status Viewed"
    elif activity == "The status of the submission has been updated.":
        return "Assignment Status Updated"
    elif activity == "A submission has been submitted.":
        return "Assignment Submitted"
    elif activity == "A file has been uploaded.":
        return "Assignment File Uploaded"
    elif activity == "Submission created.":
        return "Assignment Submission Created"
    elif activity == "Submission updated.":
        return "Assignment Submission Updated"
    elif activity == "Submission removed.":
        return "Assignment Submission Removed"
    elif activity == "Remove submission confirmation viewed.":
        return "Assignment Remove Confirmation Viewed"

    # Grade / report
    elif activity == "User graded":
        return "Grade Updated"
    elif activity == "Grade item updated":
        return "Grade Item Updated"
    elif activity == "Grade user report viewed":
        return "Grade Report Viewed"
    elif activity == "Course user report viewed":
        return "Course Report Viewed"

    # User / profile
    elif activity == "User list viewed":
        return "User List Viewed"
    elif activity == "User profile viewed":
        return "User Profile Viewed"

    # Comment / forum-like
    elif activity == "Comment created":
        return "Comment Created"
    elif activity == "Comment deleted":
        return "Comment Deleted"
    elif activity == "Subscription created":
        return "Forum Subscription Created"
    elif activity == "Subscription deleted":
        return "Forum Subscription Deleted"

    # Completion / system
    elif activity == "Course activity completion updated":
        return "Activity Completion Updated"

    # Kalau ada aktivitas baru yang belum kita mapping
    else:
        return "Other Activity"

## Cell 23 — Buat fungsi kategori aktivitas

In [37]:
def map_activity_category(mapped_activity):
    if mapped_activity in ["Course Viewed"]:
        return "course"

    elif mapped_activity in ["Material Viewed"]:
        return "material"

    elif mapped_activity.startswith("Quiz"):
        return "quiz"

    elif mapped_activity.startswith("Assignment"):
        return "assignment"

    elif mapped_activity.startswith("Forum") or mapped_activity.startswith("Comment"):
        return "forum"

    elif mapped_activity in ["Activity Completion Updated"]:
        return "system"

    elif mapped_activity in ["Grade Updated", "Grade Item Updated"]:
        return "grade"

    elif mapped_activity in ["Grade Report Viewed", "Course Report Viewed", "User List Viewed", "User Profile Viewed"]:
        return "report"

    else:
        return "other"

## Cell 24 — Terapkan mapping ke event log

In [38]:
event_log_mapped = event_log_labeled.copy()

event_log_mapped["activity_mapped"] = event_log_mapped.apply(map_activity, axis=1)
event_log_mapped["activity_category"] = event_log_mapped["activity_mapped"].apply(map_activity_category)

display(event_log_mapped[[
    "case_id",
    "activity",
    "Component",
    "activity_mapped",
    "activity_category",
    "timestamp",
    "label_performance"
]].head(20))

print("Jumlah activity mentah:", event_log_mapped["activity"].nunique())
print("Jumlah activity hasil mapping:", event_log_mapped["activity_mapped"].nunique())
print("Jumlah kategori aktivitas:", event_log_mapped["activity_category"].nunique())

,case_id,activity,Component,activity_mapped,activity_category,timestamp,label_performance
0,ADITYA PUTRA PERMANA,Course viewed,System,Course Viewed,course,2025-09-18 13:30:09,Sedang
1,ADITYA PUTRA PERMANA,Course viewed,System,Course Viewed,course,2025-09-18 13:31:17,Sedang
2,ADITYA PUTRA PERMANA,Course viewed,System,Course Viewed,course,2025-09-18 13:39:55,Sedang
3,ADITYA PUTRA PERMANA,Course activity completion updated,System,Activity Completion Updated,system,2025-09-18 14:24:42,Sedang
4,ADITYA PUTRA PERMANA,Course activity completion updated,System,Activity Completion Updated,system,2025-09-18 14:24:42,Sedang
5,ADITYA PUTRA PERMANA,Course module viewed,File,Material Viewed,material,2025-09-18 14:24:42,Sedang
6,ADITYA PUTRA PERMANA,Course viewed,System,Course Viewed,course,2025-09-18 15:41:24,Sedang
7,ADITYA PUTRA PERMANA,Course module viewed,File,Material Viewed,material,2025-09-18 15:41:57,Sedang
8,ADITYA PUTRA PERMANA,Course viewed,System,Course Viewed,course,2025-09-18 16:40:55,Sedang
9,ADITYA PUTRA PERMANA,Course viewed,System,Course Viewed,course,2025-09-18 16:49:33,Sedang


Jumlah activity mentah: 29
Jumlah activity hasil mapping: 32
Jumlah kategori aktivitas: 8


## Cell 25 — Cek hasil activity mapping

In [39]:
mapped_activity_counts = (
    event_log_mapped["activity_mapped"]
    .value_counts()
    .reset_index()
)

mapped_activity_counts.columns = ["activity_mapped", "jumlah_event"]
mapped_activity_counts["persentase"] = (
    mapped_activity_counts["jumlah_event"] / len(event_log_mapped) * 100
).round(2)

display(mapped_activity_counts)

,activity_mapped,jumlah_event,persentase
0,Quiz Viewed,50802,32.54
1,Quiz Updated,49343,31.60
2,Quiz Module Viewed,10536,6.75
3,Course Viewed,10456,6.70
4,Material Viewed,7343,4.70
5,Activity Completion Updated,6286,4.03
6,Quiz Auto-saved,5365,3.44
7,Quiz Access Prevented,3521,2.25
8,Quiz Summary Viewed,2392,1.53
9,Quiz Started,2259,1.45


## Cell 26 — Cek kategori aktivitas

In [40]:
category_counts = (
    event_log_mapped["activity_category"]
    .value_counts()
    .reset_index()
)

category_counts.columns = ["activity_category", "jumlah_event"]
category_counts["persentase"] = (
    category_counts["jumlah_event"] / len(event_log_mapped) * 100
).round(2)

display(category_counts)

,activity_category,jumlah_event,persentase
0,quiz,126428,80.97
1,course,10456,6.70
2,material,7343,4.70
3,system,6286,4.03
4,assignment,2688,1.72
5,grade,2388,1.53
6,forum,334,0.21
7,report,222,0.14


## Cell 27 — Cek apakah ada Other Activity

In [41]:
other_activities = (
    event_log_mapped[event_log_mapped["activity_mapped"] == "Other Activity"]
    [["Component", "activity"]]
    .drop_duplicates()
)

print("Jumlah jenis aktivitas yang belum termapping:", len(other_activities))
display(other_activities)

Jumlah jenis aktivitas yang belum termapping: 0


,Component,activity


## Cell 28 — Buat event log process-ready

In [45]:
excluded_categories_for_process = ["grade", "report", "system"]

event_log_process_ready = event_log_mapped[
    ~event_log_mapped["activity_category"].isin(excluded_categories_for_process)
].copy()

event_log_process_ready = event_log_process_ready.sort_values(
    by=["case_id", "timestamp"]
).reset_index(drop=True)

print("Ukuran event_log_mapped:", event_log_mapped.shape)
print("Ukuran event_log_process_ready:", event_log_process_ready.shape)

print("\nJumlah mahasiswa:", event_log_process_ready["case_id"].nunique())
print("Jumlah activity_mapped unik:", event_log_process_ready["activity_mapped"].nunique())
print("Jumlah kategori aktivitas:", event_log_process_ready["activity_category"].nunique())

print("\nDistribusi kategori process-ready:")
display(event_log_process_ready["activity_category"].value_counts())

Ukuran event_log_mapped: (156145, 16)
Ukuran event_log_process_ready: (147249, 16)

Jumlah mahasiswa: 89
Jumlah activity_mapped unik: 25
Jumlah kategori aktivitas: 5

Distribusi kategori process-ready:


activity_category
quiz          126428
course         10456
material        7343
assignment      2688
forum            334
Name: count, dtype: int64

## Cell 29 — Simpan hasil mapping

In [46]:
EVENT_LOG_MAPPED_PATH = PROCESSED_DIR / "03_event_log_students_mapped.csv"
EVENT_LOG_PROCESS_READY_PATH = PROCESSED_DIR / "03_event_log_process_ready.csv"

event_log_mapped.to_csv(EVENT_LOG_MAPPED_PATH, index=False)
event_log_process_ready.to_csv(EVENT_LOG_PROCESS_READY_PATH, index=False)

mapped_activity_counts.to_csv(
    OUTPUT_TABLES_DIR / "03_mapped_activity_counts.csv",
    index=False
)

category_counts.to_csv(
    OUTPUT_TABLES_DIR / "03_activity_category_counts.csv",
    index=False
)

print("Event log mapped berhasil disimpan ke:")
print(EVENT_LOG_MAPPED_PATH)

print("\nEvent log process-ready berhasil disimpan ke:")
print(EVENT_LOG_PROCESS_READY_PATH)

print("\nTabel mapping berhasil disimpan ke:")
print(OUTPUT_TABLES_DIR)

Event log mapped berhasil disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\processed\03_event_log_students_mapped.csv

Event log process-ready berhasil disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\processed\03_event_log_process_ready.csv

Tabel mapping berhasil disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables


## Kesimpulan Preprocessing Event Log

Pada tahap preprocessing event log, raw log gabungan dari kelas IF-49-02 dan IF-49-03 difilter berdasarkan daftar mahasiswa resmi pada data nilai. Dari 165.646 event mentah dengan 96 user unik, diperoleh 156.145 event yang berasal dari 89 mahasiswa resmi. Tujuh user non-target seperti dosen, admin, guest user, dan system user dikeluarkan dari data karena tidak termasuk unit analisis penelitian.

Setiap event kemudian dibentuk ke dalam struktur event log dengan atribut utama case_id, activity, dan timestamp. Selain itu, label performa akademik mahasiswa berhasil digabungkan ke setiap event tanpa missing value. Tahap activity mapping dilakukan untuk menyederhanakan nama aktivitas LMS yang masih bersifat teknis menjadi aktivitas yang lebih mudah dianalisis, seperti Quiz Viewed, Quiz Updated, Course Viewed, Material Viewed, Assignment Viewed, dan Forum Viewed.

Setelah aktivitas administratif seperti grade dan report dikeluarkan dari data process mining, diperoleh event log process-ready sebanyak 153.535 event dari 89 mahasiswa dengan 26 aktivitas unik hasil mapping. Event log ini akan digunakan pada tahap berikutnya untuk membentuk full log, compact log, dan analisis process mining.